## 14 — Análise pós-teste, erros e figuras finais (v4)
Este notebook é somente leitura em relação aos experimentos. Ele não carrega pesos,
não executa YOLO ou Hough e não altera threshold, tiles, merging ou métricas. Seu objetivo é:
1. verificar os resultados protegidos sintético, YOLO ASTA v3 e Hough ASTA corrigido v4.1;
2. consolidar tabelas e figuras quantitativas;
3. descrever erros sintéticos por subtipo/SNR sem retreino;
4. materializar casos ASTA por regras determinísticas reaplicadas aos resultados corrigidos;
5. produzir folhas para revisão qualitativa humana antes da publicação final da análise.
Execute em CPU. Os resultados de teste são finais e não podem motivar retuning.

In [1]:
%pip install -q pandas numpy matplotlib pillow tqdm opencv-python-headless

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from tqdm.auto import tqdm
from PIL import Image, ImageDraw, ImageFont
import gc, hashlib, json, math, os, shutil, time, zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

Image.MAX_IMAGE_PIXELS=200_000_000
drive.mount('/content/drive',force_remount=True)

BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
RUN_ID='synthetic_v3_c41d5091de56'
SYNTH_EVENT_ID='29fcf556-5eb4-43fc-8d30-8e3d88edf6c9'
YOLO_ASTA_EVENT_ID='4edef981-5590-4330-8dba-bd4d14dbcdda'
HOUGH_ASTA_EVENT_ID='7d717aea-eed8-4f4b-ba63-d4b0ca51d29c'
EXPECTED_SYNTH_RESULT='878f63dbedea0f72a4b7ed77c238d49cb2b17a89cfce4f51a27ea3b4d7fa0485'
EXPECTED_ASTA_PROTOCOL='dc4c58381b1e3e7d2a24f7fdac8fc59276e53b9d900853c1191a497319e862bc'
EXPECTED_ASTA_CONFIG='81ab0148b765fd3664b185812560826b21bf680081aa6c17d37ee2b94f773137'
EXPECTED_YOLO_ASTA_RESULT='eef0f468754b195984a4888cc8c8439f12b5e9db3af94e875082272503542f7c'
EXPECTED_HOUGH_ASTA_RESULT='303502ee091dfa26649df5bef4f61f0cf8d644925fd9487f96b2085772b444d8'
EXPECTED_HOUGH_CORRECTION_CONFIG='ea65b9d12715fdfb486609605a5ae58deef43535e9f331652feadddeeed71cd7'
EXPECTED_AUDIT_MANIFEST='91bfa07ebffceaa46271a254d3862348ce2b242a06ed44fc4c07efeeb58722ca'
EXPECTED_SYNTH_MANIFEST='b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42'

ASTA_RAW=BACKUP_DIR/'data/raw/asta_real'
AUDIT_MANIFEST=BACKUP_DIR/'experiments/asta_external_v3/audit/final/manifest_auditoria_asta_v3.csv'
ASTA_ROOT=BACKUP_DIR/'experiments/asta_external_v3/inference'
YOLO_ASTA_LOCK=ASTA_ROOT/'TRAVA_EVENTO_EXTERNO_ASTA_V3.json'
YOLO_ASTA_EVENT=ASTA_ROOT/f'evento_{YOLO_ASTA_EVENT_ID}'
YOLO_ASTA_CONS=YOLO_ASTA_EVENT/'consolidado'
HOUGH_CORR_ROOT=BACKUP_DIR/'experiments/hough_asta_correction_v4'
HOUGH_ASTA_LOCK=HOUGH_CORR_ROOT/'TRAVA_EVENTO_HOUGH_ASTA_V4.json'
HOUGH_ASTA_EVENT=HOUGH_CORR_ROOT/'events'/f'evento_{HOUGH_ASTA_EVENT_ID}'
HOUGH_ASTA_CONS=HOUGH_ASTA_EVENT/'consolidado'
SYNTH_RUN=BACKUP_DIR/'data/synthetic_runs'/RUN_ID
SYNTH_EVENT=BACKUP_DIR/'experiments/yolo_protected_test_v3'/RUN_ID/f'evento_{SYNTH_EVENT_ID}'
SYNTH_CONS=SYNTH_EVENT/'consolidado'
VAL_DIR=BACKUP_DIR/'experiments/yolo_final_multiseed_v3'/RUN_ID/'avaliacao_val'
ANALYSIS_ROOT=BACKUP_DIR/'experiments/post_test_analysis_v4'/RUN_ID
DRAFT_DRIVE=ANALYSIS_ROOT/'draft'
FINAL_DRIVE=ANALYSIS_ROOT/'final'
LOCAL=Path('/content/tcc_post_test_analysis_v4')
OUTPUT_LOCAL=LOCAL/'outputs';STAGING=LOCAL/'staging'
for pasta in [OUTPUT_LOCAL,STAGING,DRAFT_DRIVE]:pasta.mkdir(parents=True,exist_ok=True)

def agora():return datetime.now().strftime('%H:%M:%S')
def sha256_arquivo(caminho,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(caminho,'rb') as arq:
        for bloco in iter(lambda:arq.read(chunk),b''):h.update(bloco)
    return h.hexdigest()
def sha256_canonico(objeto,ensure_ascii=False):
    texto=json.dumps(objeto,sort_keys=True,separators=(',',':'),ensure_ascii=ensure_ascii)
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()
def verificar_canonico(registro,campo,ensure_ascii=False):
    copia=dict(registro);observado=copia.pop(campo)
    calculado=sha256_canonico(copia,ensure_ascii=ensure_ascii)
    if calculado!=observado:raise RuntimeError(f'Hash canônico divergente: {campo}')
    return observado
def copiar_verificado(origem,destino,esperado):
    destino=Path(destino);destino.parent.mkdir(parents=True,exist_ok=True)
    if destino.is_file() and sha256_arquivo(destino)==esperado:return
    shutil.copy2(origem,destino)
    if sha256_arquivo(destino)!=esperado:raise RuntimeError(f'Hash divergente após cópia: {origem.name}')
def gravar_json(caminho,objeto):
    Path(caminho).write_text(json.dumps(objeto,indent=2,ensure_ascii=False),encoding='utf-8')

print('Notebook 14 preparado em CPU; nenhuma biblioteca de inferência foi carregada.')

Mounted at /content/drive
Notebook 14 preparado em CPU; nenhuma biblioteca de inferência foi carregada.


## 1. Gate dos insumos finais protegidos
Confirma hashes, eventos, manifests e artefatos consolidados antes de qualquer análise.
Este gate não lê imagens ASTA e não executa detectores.

In [9]:
assert YOLO_ASTA_LOCK.is_file() and HOUGH_ASTA_LOCK.is_file() and AUDIT_MANIFEST.is_file()
trava_yolo=json.loads(YOLO_ASTA_LOCK.read_text(encoding='utf-8'))
assert trava_yolo['status']=='concluido' and trava_yolo['event_id']==YOLO_ASTA_EVENT_ID
assert trava_yolo['sha256_contrato_protocolo_asta']==EXPECTED_ASTA_PROTOCOL
assert trava_yolo['sha256_config_evento']==EXPECTED_ASTA_CONFIG
assert trava_yolo['sha256_resultado_final']==EXPECTED_YOLO_ASTA_RESULT
trava_hough=json.loads(HOUGH_ASTA_LOCK.read_text(encoding='utf-8'))
assert trava_hough['status']=='concluido' and trava_hough['event_id']==HOUGH_ASTA_EVENT_ID
assert trava_hough['sha256_config_correcao']==EXPECTED_HOUGH_CORRECTION_CONFIG
assert trava_hough['sha256_resultado_final']==EXPECTED_HOUGH_ASTA_RESULT

hashes_asta=json.loads((YOLO_ASTA_CONS/'hashes_resultados_asta_v3.json').read_text(encoding='utf-8'))
assert set(hashes_asta)=={'metricas_por_frame_metodo_asta_v3.csv','bootstrap_frames_asta_v3.csv',
    'casos_revisao_qualitativa_asta_v3.csv','resumo_final_evento_asta_v3.json'}
for nome,esperado in tqdm(sorted(hashes_asta.items()),desc='Verificando consolidação YOLO ASTA v3',unit='arq'):
    assert sha256_arquivo(YOLO_ASTA_CONS/nome)==esperado
resumo_asta_yolo=json.loads((YOLO_ASTA_CONS/'resumo_final_evento_asta_v3.json').read_text(encoding='utf-8'))
assert verificar_canonico(resumo_asta_yolo,'sha256_resultado_final',ensure_ascii=False)==EXPECTED_YOLO_ASTA_RESULT
resumo_asta_hough=json.loads((HOUGH_ASTA_CONS/'resumo_final_hough_asta_v4.json').read_text(encoding='utf-8'))
assert verificar_canonico(resumo_asta_hough,'sha256_resultado_final',ensure_ascii=False)==EXPECTED_HOUGH_ASTA_RESULT
assert resumo_asta_hough['event_id']==HOUGH_ASTA_EVENT_ID
assert resumo_asta_hough['sha256_config_correcao']==EXPECTED_HOUGH_CORRECTION_CONFIG
assert resumo_asta_hough['evento_v3_preservado']==YOLO_ASTA_EVENT_ID
assert not resumo_asta_hough['yolo_executado'] and not resumo_asta_hough['busca_parametros'] and not resumo_asta_hough['recalibracao_asta']

resumo_sint_path=SYNTH_CONS/'resumo_final_yolo_test_v3.json'
assert resumo_sint_path.is_file()
resumo_sint=json.loads(resumo_sint_path.read_text(encoding='utf-8'))
assert resumo_sint['event_id']==SYNTH_EVENT_ID
assert verificar_canonico(resumo_sint,'sha256_resultado_final',ensure_ascii=True)==EXPECTED_SYNTH_RESULT
assert sha256_arquivo(SYNTH_RUN/'manifest_geracao_v3.csv')==EXPECTED_SYNTH_MANIFEST
assert sha256_arquivo(AUDIT_MANIFEST)==EXPECTED_AUDIT_MANIFEST

manifest_asta=pd.read_csv(AUDIT_MANIFEST)
assert len(manifest_asta)==178 and manifest_asta.id_asta.nunique()==178
metricas_v3=pd.read_csv(YOLO_ASTA_CONS/'metricas_por_frame_metodo_asta_v3.csv')
metricas_yolo=metricas_v3[metricas_v3.metodo=='yolo'].copy();assert len(metricas_yolo)==178
if 'cobertura_gt' not in metricas_yolo.columns:
    metricas_yolo['cobertura_gt']=metricas_yolo['recall_centerline']
metricas_hough=pd.read_csv(HOUGH_ASTA_CONS/'metricas_hough_asta_v4.csv');assert len(metricas_hough)==178
metricas_hough['metodo']='hough'
if 'cobertura_gt' not in metricas_hough.columns:
    metricas_hough['cobertura_gt']=metricas_hough['recall_centerline']
mapa_gt=(manifest_asta.set_index('id_asta').mascara_pixels_positivos.astype(int)>0).to_dict()
metricas_hough['gt_positivo']=metricas_hough.id_asta.map(mapa_gt)
assert metricas_hough.gt_positivo.notna().all()
colunas_metricas=['id_asta','metodo','gt_positivo','pred_pixels','gt_pixels','pred_hits','gt_hits',
    'precisao_centerline','recall_centerline','f1_centerline','cobertura_gt']
metricas_asta=pd.concat([metricas_yolo[colunas_metricas],metricas_hough[colunas_metricas]],ignore_index=True)
assert len(metricas_asta)==356 and set(metricas_asta.metodo)=={'yolo','hough'}
assert len(list((YOLO_ASTA_EVENT/'checkpoints/yolo').glob('*.json')))==178
assert len(list((HOUGH_ASTA_EVENT/'checkpoints').glob('*.json')))==178
for caminho in tqdm(sorted((HOUGH_ASTA_EVENT/'checkpoints').glob('*.json')),desc='Verificando checkpoints Hough v4.1',unit='frame'):
    registro=json.loads(caminho.read_text(encoding='utf-8'))
    assert verificar_canonico(registro,'sha256_checkpoint',ensure_ascii=False)==registro['sha256_checkpoint']
    assert registro['event_id']==HOUGH_ASTA_EVENT_ID and registro['sha256_config_correcao']==EXPECTED_HOUGH_CORRECTION_CONFIG

pares_gate=metricas_asta[metricas_asta.gt_positivo.astype(bool)].pivot(index='id_asta',columns='metodo',values='f1_centerline').dropna()
pares_gate['delta']=pares_gate.yolo-pares_gate.hough;categorias_gate={}
def adicionar_gate(ids,rotulo):
    for id_asta in ids:categorias_gate.setdefault(str(id_asta),[]).append(rotulo)
adicionar_gate(pares_gate.nsmallest(12,'yolo').index,'pior_yolo')
adicionar_gate(pares_gate.nsmallest(12,'hough').index,'pior_hough_corrigido')
adicionar_gate(pares_gate.nsmallest(8,'delta').index,'menor_delta_yolo_hough_corrigido')
adicionar_gate(pares_gate.nlargest(8,'delta').index,'maior_delta_yolo_hough_corrigido')
candidatos_evento=pd.DataFrame([{'id_asta':k,'categorias':'|'.join(v)} for k,v in sorted(categorias_gate.items())])

relatorio_insumos={'status':'INSUMOS_ANALISE_POS_TESTE_V4_APROVADOS',
    'yolo_asta_event_id':YOLO_ASTA_EVENT_ID,'sha256_resultado_yolo_asta':EXPECTED_YOLO_ASTA_RESULT,
    'hough_asta_event_id':HOUGH_ASTA_EVENT_ID,'sha256_resultado_hough_asta':EXPECTED_HOUGH_ASTA_RESULT,
    'sintetico_event_id':SYNTH_EVENT_ID,
    'sha256_resultado_sintetico':EXPECTED_SYNTH_RESULT,'frames_asta':178,
    'checkpoints_asta':{'yolo_v3_reutilizado':178,'hough_v4_1_corrigido':178},'casos_evento':len(candidatos_evento),
    'inferencia_executada':False,'retuning_permitido':False}
print(json.dumps(relatorio_insumos,indent=2,ensure_ascii=False))

Verificando consolidação YOLO ASTA v3:   0%|          | 0/4 [00:00<?, ?arq/s]

Verificando checkpoints Hough v4.1:   0%|          | 0/178 [00:00<?, ?frame/s]

{
  "status": "INSUMOS_ANALISE_POS_TESTE_V4_APROVADOS",
  "yolo_asta_event_id": "4edef981-5590-4330-8dba-bd4d14dbcdda",
  "sha256_resultado_yolo_asta": "eef0f468754b195984a4888cc8c8439f12b5e9db3af94e875082272503542f7c",
  "hough_asta_event_id": "7d717aea-eed8-4f4b-ba63-d4b0ca51d29c",
  "sha256_resultado_hough_asta": "303502ee091dfa26649df5bef4f61f0cf8d644925fd9487f96b2085772b444d8",
  "sintetico_event_id": "29fcf556-5eb4-43fc-8d30-8e3d88edf6c9",
  "sha256_resultado_sintetico": "878f63dbedea0f72a4b7ed77c238d49cb2b17a89cfce4f51a27ea3b4d7fa0485",
  "frames_asta": 178,
  "checkpoints_asta": {
    "yolo_v3_reutilizado": 178,
    "hough_v4_1_corrigido": 178
  },
  "casos_evento": 36,
  "inferencia_executada": false,
  "retuning_permitido": false
}


## 2. Consolidação quantitativa e análise descritiva de erros
Produz tabelas e figuras a partir de resultados já persistidos. A comparação entre
sintético e ASTA mantém os endpoints separados: detecção/OBB no sintético e centerline no
ASTA. As análises por subtipo e SNR são descritivas e não autorizam novo treino.

In [10]:
plt.style.use('seaborn-v0_8-whitegrid')
CORES={'yolo':'#1f77b4','hough':'#d62728'}

positivos_asta=metricas_asta[metricas_asta.gt_positivo.astype(bool)].copy()
pares_asta=positivos_asta.pivot(index='id_asta',columns='metodo',values='f1_centerline').dropna()
assert len(pares_asta)==177
distribuicao=[]
for metodo in ['yolo','hough']:
    valores=pares_asta[metodo].to_numpy(float)
    distribuicao.append({'metodo':metodo,'n':len(valores),'media':float(np.mean(valores)),
        'desvio_padrao':float(np.std(valores,ddof=1)),'q05':float(np.quantile(valores,.05)),
        'q25':float(np.quantile(valores,.25)),'mediana':float(np.median(valores)),
        'q75':float(np.quantile(valores,.75)),'q95':float(np.quantile(valores,.95)),
        'min':float(np.min(valores)),'max':float(np.max(valores))})
tabela_distribuicao=pd.DataFrame(distribuicao)
tabela_distribuicao.to_csv(OUTPUT_LOCAL/'distribuicao_f1_frames_asta_v4.csv',index=False)

linhas_dominios=[]
ys=resumo_sint['resultado_primario_yolo_media_seeds'];hs=resumo_sint['hough_congelado']
ya=resumo_asta_yolo['resultado_primario_yolo'];ha=resumo_asta_hough['resultado_hough_corrigido']
linhas_dominios.extend([
    {'dominio':'sintetico','endpoint':'matching OBB/geometria','metodo':'YOLO11n-OBB (média 3 seeds)',
     'precisao':ys['precisao'],'recall':ys['recall'],'f1':ys['f1'],'comparavel_diretamente_entre_dominios':False},
    {'dominio':'sintetico','endpoint':'matching OBB/geometria','metodo':'Hough congelado',
     'precisao':hs['precisao'],'recall':hs['recall'],'f1':hs['f1'],'comparavel_diretamente_entre_dominios':False},
    {'dominio':'ASTA real','endpoint':'centerline pooled','metodo':'YOLO11n-OBB seed 20260824',
     'precisao':ya['precisao_centerline_pooled'],'recall':ya['recall_centerline_pooled'],
     'f1':ya['f1_centerline_pooled'],'comparavel_diretamente_entre_dominios':False},
    {'dominio':'ASTA real','endpoint':'centerline pooled','metodo':'Hough corrigido v4.1',
     'precisao':ha['precisao_centerline_pooled'],'recall':ha['recall_centerline_pooled'],
     'f1':ha['f1_centerline_pooled'],'comparavel_diretamente_entre_dominios':False}])
tabela_dominios=pd.DataFrame(linhas_dominios)
tabela_dominios.to_csv(OUTPUT_LOCAL/'comparacao_resultados_finais_por_dominio_v4.csv',index=False)

fig,ax=plt.subplots(figsize=(8,5))
for metodo in ['yolo','hough']:
    valores=np.sort(pares_asta[metodo].to_numpy(float));ecdf=np.arange(1,len(valores)+1)/len(valores)
    ax.plot(valores,ecdf,label=metodo.upper(),color=CORES[metodo],linewidth=2)
ax.set(xlabel='F1 centerline por frame positivo',ylabel='Fração acumulada de frames',xlim=(0,1),ylim=(0,1))
ax.legend();fig.tight_layout();fig.savefig(OUTPUT_LOCAL/'ecdf_f1_centerline_asta_v4.png',dpi=180);plt.close(fig)

fig,ax=plt.subplots(figsize=(6,6));ax.scatter(pares_asta.hough,pares_asta.yolo,s=24,alpha=.65,color='#4c78a8')
ax.plot([0,1],[0,1],'--',color='black',linewidth=1);ax.set(xlim=(-.02,1),ylim=(-.02,1),
    xlabel='F1 centerline Hough',ylabel='F1 centerline YOLO')
fig.tight_layout();fig.savefig(OUTPUT_LOCAL/'dispersao_f1_yolo_hough_asta_v4.png',dpi=180);plt.close(fig)

tempos_hough=[]
for caminho in sorted((HOUGH_ASTA_EVENT/'checkpoints').glob('*.json')):
    registro=json.loads(caminho.read_text(encoding='utf-8'));tempos_hough.append(registro['tempos_segundos'])
assert len(tempos_hough)==178
tempo_hough_pre=float(np.mean([x['preprocessamento'] for x in tempos_hough]))
tempo_hough_inf=float(np.mean([x['inferencia_hough_nms_costura'] for x in tempos_hough]))
tempo_hough_total=float(np.mean([x['total_frame'] for x in tempos_hough]))
tempos=pd.DataFrame([
    {'metodo':'YOLO','preprocessamento':ya['tempo_preprocessamento_medio_s'],'inferencia':ya['tempo_inferencia_medio_s'],
     'merging':ya['tempo_merging_medio_s'],'total':ya['tempo_total_frame_medio_s']},
    {'metodo':'Hough corrigido v4.1','preprocessamento':tempo_hough_pre,'inferencia':tempo_hough_inf,
     'merging':0.0,'total':tempo_hough_total}])
tempos.to_csv(OUTPUT_LOCAL/'tempos_medios_asta_v4.csv',index=False)
fig,ax=plt.subplots(figsize=(7,4));ax.bar(tempos.metodo,tempos.total,color=[CORES['yolo'],CORES['hough']])
ax.set(ylabel='Tempo médio por frame (s)',title='Custo operacional no ASTA')
for i,v in enumerate(tempos.total):ax.text(i,v,f'{v:.1f}s',ha='center',va='bottom')
fig.tight_layout();fig.savefig(OUTPUT_LOCAL/'tempo_total_por_metodo_asta_v4.png',dpi=180);plt.close(fig)

manifest_sint=pd.read_csv(SYNTH_RUN/'manifest_geracao_v3.csv')
manifest_test=manifest_sint[manifest_sint.split=='test'].copy();assert len(manifest_test)==900
resultados_test=[]
for seed in [20260823,20260824,20260825]:
    pasta=SYNTH_EVENT/f'seed_{seed}';registro=json.loads((pasta/'conclusao_teste_seed.json').read_text(encoding='utf-8'))
    assert sha256_arquivo(pasta/'resultado_test.csv')==registro['sha256_resultado']
    tab=pd.read_csv(pasta/'resultado_test.csv');assert len(tab)==900;tab['seed']=seed;resultados_test.append(tab)
teste_long=pd.concat(resultados_test,ignore_index=True)
por_amostra=teste_long.groupby('id',as_index=False).agg(tipo=('tipo','first'),subtipo=('subtipo','first'),
    fundo_origem=('fundo_origem','first'),sha256_fundo=('sha256_fundo','first'),
    taxa_tp_seeds=('tp','mean'),fp_medio_seeds=('fp','mean'),fn_medio_seeds=('fn','mean'))
colunas_meta=[c for c in ['id','snr_pico_alvo','comprimento_px','fonte_fundo','categoria_fundo'] if c in manifest_test.columns]
por_amostra=por_amostra.merge(manifest_test[colunas_meta],on='id',how='left',validate='one_to_one')
por_amostra.to_csv(OUTPUT_LOCAL/'erros_sinteticos_por_amostra_multiseed_v3.csv',index=False)

pos=por_amostra[por_amostra.tipo=='positivo'].copy()
subtipo=pos.groupby('subtipo',as_index=False).agg(n=('id','size'),taxa_acerto_media=('taxa_tp_seeds','mean'),
    falha_todas_seeds=('taxa_tp_seeds',lambda s:int((s==0).sum())))
subtipo.to_csv(OUTPUT_LOCAL/'acerto_sintetico_por_subtipo_v3.csv',index=False)
pos['faixa_snr']=pd.cut(pos.snr_pico_alvo,bins=[3,6,10,15,20,25],include_lowest=True,duplicates='drop')
por_snr=pos.groupby('faixa_snr',observed=True).agg(n=('id','size'),taxa_acerto_media=('taxa_tp_seeds','mean'),
    falha_todas_seeds=('taxa_tp_seeds',lambda s:int((s==0).sum()))).reset_index()
por_snr['faixa_snr']=por_snr.faixa_snr.astype(str);por_snr.to_csv(OUTPUT_LOCAL/'acerto_sintetico_por_snr_v3.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].bar(subtipo.subtipo,subtipo.taxa_acerto_media,color='#4c78a8');axes[0].set(ylim=(0,1),ylabel='Taxa média de acerto (3 seeds)',title='Por perfil sintético')
axes[1].bar(por_snr.faixa_snr,por_snr.taxa_acerto_media,color='#72b7b2');axes[1].set(ylim=(0,1),title='Por faixa de SNR')
axes[1].tick_params(axis='x',rotation=30);fig.tight_layout();fig.savefig(OUTPUT_LOCAL/'acerto_sintetico_subtipo_snr_v3.png',dpi=180);plt.close(fig)

# Validação: descrição retrospectiva, sem alterar checkpoint ou threshold já congelados.
resultados_val=[]
for seed in [20260823,20260824,20260825]:
    caminho=VAL_DIR/f'resultado_val_seed_{seed}.csv'
    assert caminho.is_file();tab=pd.read_csv(caminho);assert len(tab)==900;tab['seed']=seed;resultados_val.append(tab)
val_long=pd.concat(resultados_val,ignore_index=True)
val_resumo=val_long.groupby(['seed','tipo','subtipo'],as_index=False).agg(n=('id','size'),tp=('tp','sum'),fp=('fp','sum'),fn=('fn','sum'))
val_resumo['recall']=val_resumo.tp/(val_resumo.tp+val_resumo.fn).replace(0,np.nan)
val_resumo.to_csv(OUTPUT_LOCAL/'erros_validacao_descritivos_pos_congelamento_v3.csv',index=False)

resumo_quant={'status':'CONSOLIDACAO_QUANTITATIVA_POS_TESTE_V4_CONCLUIDA',
    'event_id_yolo_asta':YOLO_ASTA_EVENT_ID,'sha256_resultado_yolo_asta':EXPECTED_YOLO_ASTA_RESULT,
    'event_id_hough_asta':HOUGH_ASTA_EVENT_ID,'sha256_resultado_hough_asta':EXPECTED_HOUGH_ASTA_RESULT,
    'event_id_sintetico':SYNTH_EVENT_ID,
    'sha256_resultado_sintetico':EXPECTED_SYNTH_RESULT,'f1_macro_asta_yolo':ya['f1_centerline_macro_frames_positivos'],
    'f1_macro_asta_hough':ha['f1_centerline_macro_frames_positivos'],
    'ic95_delta_asta':resumo_asta_hough['bootstrap_pareado_frames_positivos']['ic95_percentil']['delta_yolo_menos_hough'],
    'comparacao_direta_sintetico_asta_permitida':False,'retuning_realizado':False,
    'arquivos_quantitativos':len(list(OUTPUT_LOCAL.glob('*')))}
gravar_json(OUTPUT_LOCAL/'resumo_quantitativo_pos_teste_v4.json',resumo_quant)
print(json.dumps(resumo_quant,indent=2,ensure_ascii=False));print(tabela_dominios)

{
  "status": "CONSOLIDACAO_QUANTITATIVA_POS_TESTE_V4_CONCLUIDA",
  "event_id_yolo_asta": "4edef981-5590-4330-8dba-bd4d14dbcdda",
  "sha256_resultado_yolo_asta": "eef0f468754b195984a4888cc8c8439f12b5e9db3af94e875082272503542f7c",
  "event_id_hough_asta": "7d717aea-eed8-4f4b-ba63-d4b0ca51d29c",
  "sha256_resultado_hough_asta": "303502ee091dfa26649df5bef4f61f0cf8d644925fd9487f96b2085772b444d8",
  "event_id_sintetico": "29fcf556-5eb4-43fc-8d30-8e3d88edf6c9",
  "sha256_resultado_sintetico": "878f63dbedea0f72a4b7ed77c238d49cb2b17a89cfce4f51a27ea3b4d7fa0485",
  "f1_macro_asta_yolo": 0.5833671994488407,
  "f1_macro_asta_hough": 0.004631929293736692,
  "ic95_delta_asta": [
    0.5294040136834355,
    0.6286870417639091
  ],
  "comparacao_direta_sintetico_asta_permitida": false,
  "retuning_realizado": false,
  "arquivos_quantitativos": 18
}
     dominio                endpoint                       metodo  precisao  \
0  sintetico  matching OBB/geometria  YOLO11n-OBB (média 3 seeds)  0.953888 

## 3. Folhas qualitativas dos casos pré-especificados
A lista é reconstruída pelas regras já congeladas: 12 menores F1 YOLO, 12 menores
F1 Hough corrigido, oito menores e oito maiores deltas. O único negativo oficial é incluído por
representar toda a estratificação negativa, não por cherry-picking. Nenhum detector roda.
Cada caso é mostrado em quatro quadros separados, tanto no frame inteiro quanto em um recorte
determinístico: imagem normalizada sem sobreposição, referência oficial em verde, YOLO somente
em ciano e Hough corrigido somente em vermelho. O GT não é repetido sobre os quadros dos
detectores. Os títulos distinguem explicitamente REFERÊNCIA de PREDIÇÃO.
As predições são primeiro rasterizadas como centerlines de um pixel na resolução nativa
de 10.560 × 10.560. Só depois o mapa de ocupação é reduzido por INTER_AREA e mostrado com
compressão de densidade. Isso evita que milhares de segmentos Hough sejam artificialmente
engrossados ao serem desenhados diretamente no preview de 640 × 640.
Execute primeiro com False. Após conferir o gate quantitativo, altere apenas para True.

In [11]:
GERAR_FOLHAS_QUALITATIVAS=True

def normalizar_frame_uint8(imagem):
    hist=np.bincount(imagem.ravel(),minlength=256);ac=np.cumsum(hist);total=int(ac[-1])
    def percentil(q):return int(np.searchsorted(ac,q/100*max(total-1,1),side='right'))
    lo,hi=percentil(1.0),percentil(99.8)
    if hi<=lo:hi=min(255,lo+1)
    saida=np.clip((imagem.astype(np.float32)-lo)*255.0/(hi-lo),0,255).astype(np.uint8)
    return saida

def ler_checkpoint(metodo,id_asta):
    caminho=(YOLO_ASTA_EVENT/'checkpoints/yolo'/f'{id_asta}.json') if metodo=='yolo' else (HOUGH_ASTA_EVENT/'checkpoints'/f'{id_asta}.json')
    registro=json.loads(caminho.read_text(encoding='utf-8'))
    assert verificar_canonico(registro,'sha256_checkpoint',ensure_ascii=False)==registro['sha256_checkpoint']
    evento_esperado=YOLO_ASTA_EVENT_ID if metodo=='yolo' else HOUGH_ASTA_EVENT_ID
    assert registro['event_id']==evento_esperado and str(registro['id_asta'])==str(id_asta)
    if metodo=='yolo':assert registro['metodo']=='yolo'
    else:assert registro['sha256_config_correcao']==EXPECTED_HOUGH_CORRECTION_CONFIG
    return registro

RENDERIZACAO_PREVIEW='painel_4x2_full_crop_predicoes_isoladas_v4'
GANHO_DENSIDADE=1.8

def mapa_densidade_preview(segmentos,shape_nativa=(10560,10560),preview=640):
    h,w=shape_nativa;canvas=np.zeros((h,w),np.uint8)
    for s in segmentos:
        p1=(int(round(s['x1'])),int(round(s['y1'])));p2=(int(round(s['x2'])),int(round(s['y2'])))
        ok,q1,q2=cv2.clipLine((0,0,w,h),p1,p2)
        if ok:cv2.line(canvas,q1,q2,255,1,cv2.LINE_8)
    pixels_nativos=int(np.count_nonzero(canvas))
    densidade=cv2.resize(canvas,(preview,preview),interpolation=cv2.INTER_AREA).astype(np.float32)/255.0
    del canvas
    return densidade,pixels_nativos

def sobrepor_densidade(base_rgb,densidade,cor,ganho=GANHO_DENSIDADE,alpha_max=0.85):
    alpha=np.clip(np.sqrt(np.maximum(densidade,0.0))*ganho,0.0,alpha_max)[...,None]
    cor_arr=np.asarray(cor,dtype=np.float32).reshape(1,1,3)
    return np.clip(base_rgb.astype(np.float32)*(1.0-alpha)+cor_arr*alpha,0,255).astype(np.uint8)

def quantidade_segmentos(registro,metodo):
    if metodo=='yolo':return int(registro['segmentos_apos_merging'])
    return int(registro['segmentos_pos_costura_proprietaria'])

def calcular_recorte_deterministico(mascara,segmentos_yolo,tamanho=1000):
    h,w=mascara.shape
    linhas=np.flatnonzero(np.max(mascara,axis=1));colunas=np.flatnonzero(np.max(mascara,axis=0))
    if len(linhas) and len(colunas):
        cx=int(round((int(colunas[0])+int(colunas[-1]))/2));cy=int(round((int(linhas[0])+int(linhas[-1]))/2))
        regra='centro_bbox_referencia_oficial'
    elif segmentos_yolo:
        s=max(segmentos_yolo,key=lambda z:math.hypot(float(z['x2'])-float(z['x1']),float(z['y2'])-float(z['y1'])))
        cx=int(round((float(s['x1'])+float(s['x2']))/2));cy=int(round((float(s['y1'])+float(s['y2']))/2))
        regra='centro_maior_segmento_yolo_no_negativo'
    else:
        cx=w//2;cy=h//2;regra='centro_geometrico_sem_predicao_yolo'
    lado=min(int(tamanho),h,w);x0=max(0,min(w-lado,cx-lado//2));y0=max(0,min(h-lado,cy-lado//2))
    return (int(x0),int(y0),int(x0+lado),int(y0+lado)),regra

def mapa_densidade_recorte(segmentos,recorte,preview=480):
    x0,y0,x1,y1=recorte;h=y1-y0;w=x1-x0;canvas=np.zeros((h,w),np.uint8);visiveis=0
    for s in segmentos:
        p1=(int(round(s['x1']))-x0,int(round(s['y1']))-y0);p2=(int(round(s['x2']))-x0,int(round(s['y2']))-y0)
        ok,q1,q2=cv2.clipLine((0,0,w,h),p1,p2)
        if ok:cv2.line(canvas,q1,q2,255,1,cv2.LINE_8);visiveis+=1
    pixels=int(np.count_nonzero(canvas))
    densidade=cv2.resize(canvas,(preview,preview),interpolation=cv2.INTER_AREA).astype(np.float32)/255.0
    return densidade,pixels,visiveis

def sha256_segmentos(segmentos):
    texto=json.dumps(segmentos,sort_keys=True,separators=(',',':'),ensure_ascii=False)
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()

# Regressão: a rasterização conserva a centerline de um pixel antes da redução.
_dens_teste,_px_teste=mapa_densidade_preview(
    [{'x1':8,'y1':32,'x2':119,'y2':32}],shape_nativa=(128,128),preview=64)
assert _px_teste==112 and _dens_teste.shape==(64,64) and 0<float(_dens_teste.max())<=1.0
del _dens_teste
_base_teste=np.full((8,8,3),100,np.uint8);_zero_teste=np.zeros((8,8),np.float32);_gt_teste=np.zeros((8,8),np.float32);_gt_teste[3:5,3:5]=1
_painel_gt_teste=sobrepor_densidade(_base_teste,_gt_teste,(0,255,80))
_painel_y_zero=sobrepor_densidade(_base_teste,_zero_teste,(0,220,255))
_painel_h_zero=sobrepor_densidade(_base_teste,_zero_teste,(255,60,40))
assert not np.array_equal(_painel_gt_teste,_base_teste)
assert np.array_equal(_painel_y_zero,_base_teste) and np.array_equal(_painel_h_zero,_base_teste)
_um=np.ones((1,1),np.float32);_preto=np.zeros((1,1,3),np.uint8)
_ciano=sobrepor_densidade(_preto,_um,(0,220,255))[0,0];_verde=sobrepor_densidade(_preto,_um,(0,255,80))[0,0];_vermelho=sobrepor_densidade(_preto,_um,(255,60,40))[0,0]
assert _ciano[2]>_ciano[0] and _verde[1]>_verde[2] and _vermelho[0]>_vermelho[1]
del _base_teste,_zero_teste,_gt_teste,_painel_gt_teste,_painel_y_zero,_painel_h_zero,_um,_preto

if not GERAR_FOLHAS_QUALITATIVAS:
    print('Folhas qualitativas bloqueadas. Confira CONSOLIDACAO_QUANTITATIVA_POS_TESTE_V4_CONCLUIDA e altere somente a chave para True.')
else:
    pares=metricas_asta[metricas_asta.gt_positivo.astype(bool)].pivot(index='id_asta',columns='metodo',values='f1_centerline').reset_index()
    categorias={}
    def adicionar(ids,rotulo):
        for id_asta in ids:categorias.setdefault(str(id_asta),[]).append(rotulo)
    positivos=pares.dropna(subset=['yolo','hough']).copy();positivos['delta']=positivos.yolo-positivos.hough
    adicionar(positivos.nsmallest(12,'yolo').id_asta,'pior_yolo')
    adicionar(positivos.nsmallest(12,'hough').id_asta,'pior_hough_corrigido')
    adicionar(positivos.nsmallest(8,'delta').id_asta,'menor_delta_yolo_hough_corrigido')
    adicionar(positivos.nlargest(8,'delta').id_asta,'maior_delta_yolo_hough_corrigido')
    assert set(candidatos_evento.id_asta.astype(str))==set(categorias)
    negativos=manifest_asta[manifest_asta.mascara_pixels_positivos.astype(int)==0]
    assert len(negativos)==1;id_neg=str(negativos.iloc[0].id_asta);categorias[id_neg]=['unico_negativo_oficial']
    ids=sorted(categorias);PREVIEW=480
    paineis=OUTPUT_LOCAL/'paineis_qualitativos';folhas=OUTPUT_LOCAL/'folhas_qualitativas'
    paineis.mkdir(exist_ok=True);folhas.mkdir(exist_ok=True)
    try:
        fonte=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',18)
        fonte_titulo=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',14)
    except Exception:fonte=fonte_titulo=ImageFont.load_default()
    indice=[]
    mapa=manifest_asta.set_index('id_asta')
    for id_asta in tqdm(ids,desc='Gerando painéis qualitativos',unit='frame'):
        linha=mapa.loc[id_asta];img_local=STAGING/linha.arquivo_imagem;mask_local=STAGING/linha.arquivo_mascara
        copiar_verificado(ASTA_RAW/linha.arquivo_imagem,img_local,linha.sha256_imagem)
        copiar_verificado(ASTA_RAW/linha.arquivo_mascara,mask_local,linha.sha256_mascara)
        imagem=cv2.imread(str(img_local),cv2.IMREAD_GRAYSCALE);mascara=cv2.imread(str(mask_local),cv2.IMREAD_GRAYSCALE)
        assert imagem is not None and mascara is not None and imagem.shape==mascara.shape==(10560,10560)
        norm=normalizar_frame_uint8(imagem)
        pequena=cv2.resize(norm,(PREVIEW,PREVIEW),interpolation=cv2.INTER_AREA)
        mask_density=cv2.resize(mascara,(PREVIEW,PREVIEW),interpolation=cv2.INTER_AREA).astype(np.float32)/255.0
        base=cv2.cvtColor(pequena,cv2.COLOR_GRAY2RGB)
        gt=sobrepor_densidade(base,mask_density,(0,255,80),ganho=1.8,alpha_max=0.65)
        ry=ler_checkpoint('yolo',id_asta);rh=ler_checkpoint('hough',id_asta)
        dens_y,pixels_y_render=mapa_densidade_preview(ry['segmentos'],preview=PREVIEW)
        dens_h,pixels_h_render=mapa_densidade_preview(rh['segmentos'],preview=PREVIEW)
        assert pixels_y_render==int(ry['metricas']['pred_pixels'])
        assert pixels_h_render==int(rh['metricas']['pred_pixels'])
        # Predições são mostradas sobre a imagem limpa, nunca sobre o GT.
        painel_y=sobrepor_densidade(base,dens_y,(0,220,255))
        painel_h=sobrepor_densidade(base,dens_h,(255,60,40))
        recorte,regra_recorte=calcular_recorte_deterministico(mascara,ry['segmentos'],tamanho=1000)
        x0,y0,x1,y1=recorte
        base_crop=cv2.cvtColor(cv2.resize(norm[y0:y1,x0:x1],(PREVIEW,PREVIEW),interpolation=cv2.INTER_AREA),cv2.COLOR_GRAY2RGB)
        mask_crop=cv2.resize(mascara[y0:y1,x0:x1],(PREVIEW,PREVIEW),interpolation=cv2.INTER_AREA).astype(np.float32)/255.0
        gt_crop=sobrepor_densidade(base_crop,mask_crop,(0,255,80),ganho=1.8,alpha_max=0.65)
        dens_y_crop,pixels_y_crop,n_y_crop=mapa_densidade_recorte(ry['segmentos'],recorte,preview=PREVIEW)
        dens_h_crop,pixels_h_crop,n_h_crop=mapa_densidade_recorte(rh['segmentos'],recorte,preview=PREVIEW)
        painel_y_crop=sobrepor_densidade(base_crop,dens_y_crop,(0,220,255))
        painel_h_crop=sobrepor_densidade(base_crop,dens_h_crop,(255,60,40))
        CABECALHO=90;ALTURA_TITULO=42;ALTURA_PAINEL=CABECALHO+2*(ALTURA_TITULO+PREVIEW)
        canvas=Image.new('RGB',(PREVIEW*4,ALTURA_PAINEL),'white');draw=ImageDraw.Draw(canvas)
        fy=ry['metricas']['f1_centerline'];fh=rh['metricas']['f1_centerline']
        fy_txt='NA' if fy is None else f'{float(fy):.4f}';fh_txt='NA' if fh is None else f'{float(fh):.4f}'
        n_y=quantidade_segmentos(ry,'yolo');n_h=quantidade_segmentos(rh,'hough')
        texto=f"{id_asta} | {','.join(categorias[id_asta])}"
        texto_metricas=f"YOLO: F1={fy_txt}, n={n_y} | Hough corrigido v4.1: F1={fh_txt}, n={n_h} | recorte={recorte}"
        y_titulo_1=CABECALHO;y_imagem_1=CABECALHO+ALTURA_TITULO
        y_titulo_2=y_imagem_1+PREVIEW;y_imagem_2=y_titulo_2+ALTURA_TITULO
        draw.text((12,8),texto,fill='black',font=fonte)
        draw.text((12,38),texto_metricas,fill='black',font=fonte)
        titulos_full=['FRAME INTEIRO | imagem limpa','REFERÊNCIA OFICIAL (verde) | NÃO É DETECÇÃO',
            'YOLO (ciano) | SOMENTE PREDIÇÃO, SEM MÁSCARA','HOUGH v4.1 (vermelho) | SOMENTE PREDIÇÃO, SEM MÁSCARA']
        titulos_crop=['RECORTE | imagem limpa','RECORTE | REFERÊNCIA OFICIAL, NÃO É DETECÇÃO',
            f'RECORTE | YOLO, SÓ PREDIÇÃO (n visível={n_y_crop})',f'RECORTE | HOUGH v4.1, SÓ PREDIÇÃO (n visível={n_h_crop})']
        for j,titulo in enumerate(titulos_full):draw.text((j*PREVIEW+8,y_titulo_1+10),titulo,fill='black',font=fonte_titulo)
        for j,titulo in enumerate(titulos_crop):draw.text((j*PREVIEW+8,y_titulo_2+10),titulo,fill='black',font=fonte_titulo)
        for j,q in enumerate([base,gt,painel_y,painel_h]):canvas.paste(Image.fromarray(q),(j*PREVIEW,y_imagem_1))
        for j,q in enumerate([base_crop,gt_crop,painel_y_crop,painel_h_crop]):canvas.paste(Image.fromarray(q),(j*PREVIEW,y_imagem_2))
        if n_y==0:
            assert np.array_equal(painel_y,base) and np.array_equal(painel_y_crop,base_crop)
        if n_h==0:
            assert np.array_equal(painel_h,base) and np.array_equal(painel_h_crop,base_crop)
        painel_path=paineis/f'{id_asta}.jpg';canvas.save(painel_path,quality=90,optimize=True)
        indice.append({'id_asta':id_asta,'categorias':'|'.join(categorias[id_asta]),
            'f1_yolo':fy,'f1_hough_corrigido':fh,'segmentos_yolo':n_y,
            'segmentos_hough_corrigido':n_h,'pred_pixels_yolo':pixels_y_render,
            'pred_pixels_hough':pixels_h_render,'sha256_checkpoint_yolo':ry['sha256_checkpoint'],
            'sha256_checkpoint_hough':rh['sha256_checkpoint'],'sha256_segmentos_yolo':sha256_segmentos(ry['segmentos']),
            'sha256_segmentos_hough':sha256_segmentos(rh['segmentos']),
            'recorte_x0':x0,'recorte_y0':y0,'recorte_x1':x1,'recorte_y1':y1,'regra_recorte':regra_recorte,
            'pixels_yolo_no_recorte':pixels_y_crop,'pixels_hough_no_recorte':pixels_h_crop,
            'segmentos_yolo_visiveis_recorte':n_y_crop,'segmentos_hough_visiveis_recorte':n_h_crop,
            'renderizacao_preview':RENDERIZACAO_PREVIEW,'painel':painel_path.name,
            'observacao_humana':'','padrao_erro':'','aprovado_para_figura_tcc':''})
        for p in [img_local,mask_local]:
            if p.exists():p.unlink()
        del imagem,mascara,norm,pequena,mask_density,base,gt,dens_y,dens_h,painel_y,painel_h,base_crop,mask_crop,gt_crop,dens_y_crop,dens_h_crop,painel_y_crop,painel_h_crop,ry,rh,canvas;gc.collect()

    indice_df=pd.DataFrame(indice)
    assert indice_df.sha256_checkpoint_yolo.nunique()==len(indice_df)
    assert indice_df.sha256_checkpoint_hough.nunique()==len(indice_df)
    assert indice_df.sha256_segmentos_hough.nunique()==len(indice_df), 'Predições Hough idênticas entre casos distintos.'
    indice_path=OUTPUT_LOCAL/'indice_revisao_qualitativa_asta_v4.csv';indice_df.to_csv(indice_path,index=False)
    por_folha=2;ALTURA_PAINEL=90+2*(42+PREVIEW)
    for inicio in range(0,len(indice_df),por_folha):
        lote=indice_df.iloc[inicio:inicio+por_folha];imgs=[Image.open(paineis/x.painel).convert('RGB') for x in lote.itertuples()]
        folha=Image.new('RGB',(PREVIEW*4,ALTURA_PAINEL*len(imgs)),'white')
        for i,img in enumerate(imgs):folha.paste(img,(0,i*ALTURA_PAINEL))
        folha.save(folhas/f'folha_{inicio//por_folha+1:02d}.jpg',quality=90,optimize=True)
        for img in imgs:img.close()

    zip_path=OUTPUT_LOCAL/'folhas_revisao_qualitativa_asta_v4.zip'
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(paineis.glob('*.jpg'))+sorted(folhas.glob('*.jpg')):z.write(p,p.relative_to(OUTPUT_LOCAL))
        z.write(indice_path,indice_path.name)
    arquivos=[p for p in OUTPUT_LOCAL.rglob('*') if p.is_file() and p.name!='hashes_artefatos_analise_v4.json']
    hashes={str(p.relative_to(OUTPUT_LOCAL)).replace('\\','/'):sha256_arquivo(p) for p in sorted(arquivos)}
    gravar_json(OUTPUT_LOCAL/'hashes_artefatos_analise_v4.json',hashes)
    for p in tqdm(sorted(OUTPUT_LOCAL.rglob('*')),desc='Publicando rascunho pós-teste',unit='arq'):
        if p.is_file():
            alvo=DRAFT_DRIVE/p.relative_to(OUTPUT_LOCAL);alvo.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,alvo)
    for nome,esperado in hashes.items():assert sha256_arquivo(DRAFT_DRIVE/nome)==esperado
    relatorio={'status':'ANALISE_POS_TESTE_V4_PRONTA_PARA_REVISAO',
        'event_id_yolo_asta':YOLO_ASTA_EVENT_ID,'sha256_resultado_yolo_asta':EXPECTED_YOLO_ASTA_RESULT,
        'event_id_hough_asta':HOUGH_ASTA_EVENT_ID,'sha256_resultado_hough_asta':EXPECTED_HOUGH_ASTA_RESULT,
        'casos_evento':len(candidatos_evento),
        'negativos_incluidos':1,'casos_totais':len(indice_df),'folhas':len(list(folhas.glob('*.jpg'))),
        'sha256_zip_folhas':sha256_arquivo(zip_path),'sha256_indice':sha256_arquivo(indice_path),
        'renderizacao_preview':RENDERIZACAO_PREVIEW,'espessura_centerline_nativa_px':1,
        'composicao_painel':'4x2: frame inteiro e recorte deterministico; imagem limpa | referencia verde | YOLO ciano sem GT | Hough v4.1 vermelho sem GT',
        'regra_recorte_positivos':'centro_bbox_referencia_oficial','regra_recorte_negativo':'centro_maior_segmento_yolo; fallback centro_geometrico',
        'checkpoints_yolo_unicos':int(indice_df.sha256_checkpoint_yolo.nunique()),
        'checkpoints_hough_unicos':int(indice_df.sha256_checkpoint_hough.nunique()),
        'predicoes_hough_unicas':int(indice_df.sha256_segmentos_hough.nunique()),
        'predicoes_yolo_unicas':int(indice_df.sha256_segmentos_yolo.nunique()),
        'reducao_preview':'INTER_AREA','transformacao_alpha':'clip(sqrt(densidade)*1.8,0,0.85)',
        'artefatos_rascunho':len(hashes),'inferencia_executada':False,'retuning_realizado':False,
        'publicacao_final_aprovada':False}
    gravar_json(OUTPUT_LOCAL/'relatorio_rascunho_analise_v4.json',relatorio)
    shutil.copy2(OUTPUT_LOCAL/'relatorio_rascunho_analise_v4.json',DRAFT_DRIVE/'relatorio_rascunho_analise_v4.json')
    print(json.dumps(relatorio,indent=2,ensure_ascii=False))
    print('Envie folhas_revisao_qualitativa_asta_v4.zip e o relatório antes da publicação final.')

Gerando painéis qualitativos:   0%|          | 0/37 [00:00<?, ?frame/s]

Publicando rascunho pós-teste:   0%|          | 0/74 [00:00<?, ?arq/s]

{
  "status": "ANALISE_POS_TESTE_V4_PRONTA_PARA_REVISAO",
  "event_id_yolo_asta": "4edef981-5590-4330-8dba-bd4d14dbcdda",
  "sha256_resultado_yolo_asta": "eef0f468754b195984a4888cc8c8439f12b5e9db3af94e875082272503542f7c",
  "event_id_hough_asta": "7d717aea-eed8-4f4b-ba63-d4b0ca51d29c",
  "sha256_resultado_hough_asta": "303502ee091dfa26649df5bef4f61f0cf8d644925fd9487f96b2085772b444d8",
  "casos_evento": 36,
  "negativos_incluidos": 1,
  "casos_totais": 37,
  "folhas": 19,
  "sha256_zip_folhas": "921b517ea5aa8682b6a5014fc3b7d84061c4bb36333334c6eeaaa30b3a3bffc1",
  "sha256_indice": "30da046b2bfba6939400c5d0f943d016ce003eec206c82dbe825a95c6ad2adb4",
  "renderizacao_preview": "painel_4x2_full_crop_predicoes_isoladas_v4",
  "espessura_centerline_nativa_px": 1,
  "composicao_painel": "4x2: frame inteiro e recorte deterministico; imagem limpa | referencia verde | YOLO ciano sem GT | Hough v4.1 vermelho sem GT",
  "regra_recorte_positivos": "centro_bbox_referencia_oficial",
  "regra_recorte_neg

In [ ]:
rh = ler_checkpoint('hough','ML1_20220525_195141_red')
segs = rh['segmentos']
print(f"Total de segmentos: {len(segs):,}")

import numpy as np, math
angs = np.array([math.degrees(math.atan2(s['y2']-s['y1'], s['x2']-s['x1'])) % 180 for s in segs])
print(f"A menos de 5° de um eixo: {np.mean((angs<5)|(angs>175)|(abs(angs-90)<5)):.1%} (uniforme seria 5,6%)")

# testa alinhamento a bordas de tile: posicao dos extremos modulo o stride
xs = np.array([s['x1'] for s in segs] + [s['x2'] for s in segs])
resto = xs % 384
hist,_ = np.histogram(resto, bins=24, range=(0,384))
print("Extremos por posição dentro do tile (24 bins de 16px):")
print(hist)
print(f"Razão pico/mediana: {hist.max()/np.median(hist):.2f}  (>3 indicaria alinhamento a borda de tile)")

Total de segmentos: 7,786
A menos de 5° de um eixo: 4.4% (uniforme seria 5,6%)
Extremos por posição dentro do tile (24 bins de 16px):
[1480 1029 1053 1078 1095 1104  948 1459  500  479  413  338  346  350
  364  336  369  351  349  346  370  412  460  543]
Razão pico/mediana: 3.39  (>3 indicaria alinhamento a borda de tile)


## 4. Publicação após revisão humana
Só execute depois de compartilhar e revisar as folhas. A publicação não altera resultados;
apenas registra a interpretação humana e congela tabelas/figuras pós-teste.

In [3]:
APROVAR_PUBLICACAO_ANALISE=True

NOTA_REVISAO_HUMANA=(
    'Foram revisadas 19 folhas referentes a 37 casos, incluindo sucessos, '
    'falhas, casos intermediários, múltiplas trilhas e o único negativo oficial. '
    'Confirmaram-se o alinhamento imagem-máscara, os recortes determinísticos '
    'e o isolamento visual entre referência oficial, YOLO e Hough v4.1. '
    'A saturação vermelha do Hough representa suas predições tile-wise e é '
    'coerente com a alta cobertura acompanhada de precisão extremamente baixa. '
    'Aprovado para consolidação documental, sem nova inferência ou retuning.'
)

if not APROVAR_PUBLICACAO_ANALISE:
    print('Publicação final bloqueada. Revise primeiro as folhas qualitativas.')
else:
    if NOTA_REVISAO_HUMANA.strip() in {'','Unknown / to be confirmed'}:
        raise RuntimeError('Registre uma síntese objetiva da revisão humana antes de publicar.')
    hashes_path=DRAFT_DRIVE/'hashes_artefatos_analise_v4.json'
    relatorio_path=DRAFT_DRIVE/'relatorio_rascunho_analise_v4.json'
    assert hashes_path.is_file() and relatorio_path.is_file()
    hashes=json.loads(hashes_path.read_text(encoding='utf-8'))
    relatorio=json.loads(relatorio_path.read_text(encoding='utf-8'))
    assert relatorio['sha256_resultado_yolo_asta']==EXPECTED_YOLO_ASTA_RESULT
    assert relatorio['sha256_resultado_hough_asta']==EXPECTED_HOUGH_ASTA_RESULT and not relatorio['inferencia_executada']
    # Versões anteriores podiam incluir o próprio relatório no manifesto antes de
    # sobrescrevê-lo com o conteúdo final, tornando esse único hash autocontraditório.
    # Arquivos de controle são validados semanticamente abaixo; os artefatos científicos
    # permanecem integralmente verificados pelo manifesto.
    controles={'hashes_artefatos_analise_v4.json','relatorio_rascunho_analise_v4.json'}
    hashes_artefatos={nome:esperado for nome,esperado in hashes.items() if Path(nome).name not in controles}
    divergencias=[]
    for nome,esperado in tqdm(sorted(hashes_artefatos.items()),desc='Verificando rascunho aprovado',unit='arq'):
        atual=sha256_arquivo(DRAFT_DRIVE/nome)
        if atual!=esperado:divergencias.append({'arquivo':nome,'esperado':esperado,'atual':atual})
    if divergencias:
        raise RuntimeError(f'Artefatos científicos divergentes no rascunho: {divergencias[:5]}')
    assert sha256_arquivo(DRAFT_DRIVE/'folhas_revisao_qualitativa_asta_v4.zip')==relatorio['sha256_zip_folhas']
    assert sha256_arquivo(DRAFT_DRIVE/'indice_revisao_qualitativa_asta_v4.csv')==relatorio['sha256_indice']
    if (FINAL_DRIVE/'resumo_final_analise_pos_teste_v4.json').exists():
        raise RuntimeError('Análise pós-teste já publicada; sobrescrita é proibida.')
    for p in tqdm(sorted(DRAFT_DRIVE.rglob('*')),desc='Publicando análise final',unit='arq'):
        if p.is_file():
            alvo=FINAL_DRIVE/p.relative_to(DRAFT_DRIVE);alvo.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,alvo)
    resumo_final={'protocolo':'post_test_analysis_v4','status':'ANALISE_POS_TESTE_V4_PUBLICADA',
        'publicado_em_utc':datetime.now(timezone.utc).isoformat(),
        'event_id_yolo_asta':YOLO_ASTA_EVENT_ID,'sha256_resultado_yolo_asta':EXPECTED_YOLO_ASTA_RESULT,
        'event_id_hough_asta':HOUGH_ASTA_EVENT_ID,'sha256_resultado_hough_asta':EXPECTED_HOUGH_ASTA_RESULT,
        'event_id_sintetico':SYNTH_EVENT_ID,
        'sha256_resultado_sintetico':EXPECTED_SYNTH_RESULT,'revisao_humana_aprovada':True,
        'nota_revisao_humana':NOTA_REVISAO_HUMANA.strip(),'casos_revisados':relatorio['casos_totais'],
        'inferencia_executada':False,'retuning_realizado':False,'resultados_experimentais_alterados':False}
    resumo_final['sha256_analise_final']=sha256_canonico(resumo_final,ensure_ascii=False)
    gravar_json(FINAL_DRIVE/'resumo_final_analise_pos_teste_v4.json',resumo_final)
    print(json.dumps(resumo_final,indent=2,ensure_ascii=False))
    print('ANÁLISE PÓS-TESTE PUBLICADA. Resultados experimentais permanecem imutáveis.')


Verificando rascunho aprovado:   0%|          | 0/70 [00:00<?, ?arq/s]

Publicando análise final:   0%|          | 0/74 [00:00<?, ?arq/s]

{
  "protocolo": "post_test_analysis_v4",
  "status": "ANALISE_POS_TESTE_V4_PUBLICADA",
  "publicado_em_utc": "2026-09-03T17:33:53.704357+00:00",
  "event_id_yolo_asta": "4edef981-5590-4330-8dba-bd4d14dbcdda",
  "sha256_resultado_yolo_asta": "eef0f468754b195984a4888cc8c8439f12b5e9db3af94e875082272503542f7c",
  "event_id_hough_asta": "7d717aea-eed8-4f4b-ba63-d4b0ca51d29c",
  "sha256_resultado_hough_asta": "303502ee091dfa26649df5bef4f61f0cf8d644925fd9487f96b2085772b444d8",
  "event_id_sintetico": "29fcf556-5eb4-43fc-8d30-8e3d88edf6c9",
  "sha256_resultado_sintetico": "878f63dbedea0f72a4b7ed77c238d49cb2b17a89cfce4f51a27ea3b4d7fa0485",
  "revisao_humana_aprovada": true,
  "nota_revisao_humana": "Foram revisadas 19 folhas referentes a 37 casos, incluindo sucessos, falhas, casos intermediários, múltiplas trilhas e o único negativo oficial. Confirmaram-se o alinhamento imagem-máscara, os recortes determinísticos e o isolamento visual entre referência oficial, YOLO e Hough v4.1. A saturação ve